# 02 - Label construction, draws, and train/test split

Builds on the EDA findings:
- `score`/`mate` are complementary and need unifying into one continuous eval.
- Outcome (`white_result`) is a string: `'1'`, `'0'`, `'1/2'` — draws are ~5.5% of positions.
- ~16.6K games, ~66 positions/game on average — splitting by row would leak correlated positions across train/test, so we split by `game_id`.

The reusable logic lives in [`src/data.py`](../src/data.py) so it isn't duplicated across notebooks; this notebook just calls it and sanity-checks the result.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.data import unify_score, add_outcome_label, game_level_split

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/raw/positions.csv', low_memory=False)
df.shape

(1091078, 17)

## Unify `score` / `mate`
One continuous `eval_cp` column: centipawns, capped at +/-2000, with forced mates mapped beyond that range (shorter mate = more extreme).

In [2]:
df['eval_cp'] = unify_score(df)

# sanity check: sign should track which side is better; magnitude should be bounded
print(df['eval_cp'].describe())
df[['score', 'mate', 'eval_cp']].sample(10, random_state=1)

count    1.091078e+06
mean     1.343267e+02
std      3.124043e+03
min     -9.999000e+03
25%     -1.650000e+02
50%      8.000000e+00
75%      2.710000e+02
max      9.999000e+03
Name: eval_cp, dtype: float64


,score,mate,eval_cp
982217,-1071.0,NaN,-1071.0
1032314,NaN,-4.0,-9996.0
145767,NaN,11.0,9989.0
421155,-971.0,NaN,-971.0
125961,571.0,NaN,571.0
226287,25.0,NaN,25.0
997271,-108.0,NaN,-108.0
620875,18.0,NaN,18.0
269532,-1180.0,NaN,-1180.0
855430,503.0,NaN,503.0


## Outcome label + draws
`add_outcome_label` builds `white_win` (1/0) from `white_result` and, by default, drops draws — that's our binary classification target for now.

In [3]:
rows_before = len(df)
df = add_outcome_label(df, drop_draws=True)

print(f'dropped {rows_before - len(df):,} draw rows ({(rows_before - len(df)) / rows_before:.1%})')
print(f'remaining rows: {len(df):,}')
df['white_win'].value_counts(normalize=True)

dropped 59,764 draw rows (5.5%)
remaining rows: 1,031,314


white_win
1.0    0.517926
0.0    0.482074
Name: proportion, dtype: float64

## Game-level train/test split
Split on `game_id` (not row) so no game's positions span both sides, stratified on each game's outcome to keep class balance similar in train and test.

In [4]:
train_df, test_df = game_level_split(df, test_size=0.2, random_state=42)

print(f'train: {len(train_df):,} rows, {train_df["game_id"].nunique():,} games')
print(f'test:  {len(test_df):,} rows, {test_df["game_id"].nunique():,} games')

# leakage check: no game_id should appear in both splits
overlap = set(train_df['game_id']) & set(test_df['game_id'])
print(f'overlapping game_ids: {len(overlap)}')

print('\ntrain white_win rate:', train_df['white_win'].mean())
print('test  white_win rate:', test_df['white_win'].mean())

train: 822,936 rows, 12,897 games
test:  208,378 rows, 3,225 games
overlapping game_ids: 0

train white_win rate: 0.5189103405368096
test  white_win rate: 0.5140369904692434


## Save processed splits
So the modeling notebook (baseline logistic regression) can load these directly instead of redoing this prep.

In [5]:
train_df.to_csv('../data/processed/train.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)
print('saved.')

saved.
